# Fit and evaluate a spam filter

The UCI Spambase benchmark has 4,601 labeled emails. Here we use the full dataset and all 57 input measurements to compare a single decision tree, bootstrap aggregation, and a random forest. All four models are evaluated on the same held-out messages.

**Start from the notebook outline.** Open **Follow one email through the tuned forest** for a visual walk-through, **Change one setting at a time** for tuning, or **Evaluate the selected forest** for held-out results. Run all cells first so the interactive cards are ready.

This is an evaluation on a historical benchmark. The source contains measurements and labels and does not include the original message text. The distribution reflects the contributors' own email sources, so the numbers apply to this benchmark and split.

**Data:** Spambase by Mark Hopkins, Erik Reeber, George Forman, and Jaap Suermondt (1999). [UCI Machine Learning Repository, DOI 10.24432/C53G6X](https://doi.org/10.24432/C53G6X) · [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) · [Original field definitions](https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.names)

In [ ]:
import os

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, recall_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold, validation_curve
from sklearn.tree import DecisionTreeClassifier

print(f'NumPy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}')

## Read the benchmark and set aside the test emails

The input columns record the frequency of 48 words, the frequency of six punctuation characters, and three measurements about runs of capital letters. For example, `char_freq_!` is the percentage of all characters in the message that are exclamation marks. The label `spam` is **1** for spam and **0** for legitimate email as labeled by the dataset contributors.

We keep the benchmark's original row numbers as IDs. These IDs let us trace predictions and are excluded from the input features.

Some messages have identical measurements. Keep every source row, but group identical feature vectors so they cannot appear on both sides of a split. A stratified group split also aims to keep the spam proportions roughly similar. Set aside one of four folds as the final test set, then use new grouped folds within the training partition for tuning.

In [ ]:
seed = 37
spambase_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/spambase/spambase.data'
words = (
    'make address all 3d our over remove internet order mail receive will people report addresses free '
    'business email you credit your font 000 money hp hpl george 650 lab labs telnet 857 data 415 85 '
    'technology 1999 parts pm direct cs meeting original project re edu table conference'
).split()
feature_names = (
    [f'word_freq_{word}' for word in words]
    + [f'char_freq_{char}' for char in ';([!$#']
    + ['capital_run_length_average', 'capital_run_length_longest', 'capital_run_length_total']
)
source = os.environ.get('UCI_SPAMBASE_FILE', spambase_url)
email_records = pd.read_csv(source, header=None, names=feature_names + ['spam'])
email_records.index = pd.Index([f'E{i}' for i in range(1, len(email_records) + 1)], name='UCI row ID')
X = email_records[feature_names]
y = email_records['spam']
exemplar_columns = ['char_freq_$', 'char_freq_!', 'word_freq_free', 'capital_run_length_longest']
groups = X.groupby(feature_names, sort=False, dropna=False).ngroup()
test_split = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
train_rows, test_rows = next(test_split.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_rows], X.iloc[test_rows]
y_train, y_test = y.iloc[train_rows], y.iloc[test_rows]
train_groups, test_groups = groups.iloc[train_rows], groups.iloc[test_rows]

display(X_train[exemplar_columns].assign(spam=y_train).head())
print(f'{len(X_train):,} training rows | {len(X_test):,} held-out rows')
print(f'Recorded spam share: training {y_train.mean():.1%} | held-out {y_test.mean():.1%}')

## Check the email records

Check the number of fields and rows, the labels, missing values, and identical feature rows. A repeated feature vector need not be a repeated email, so we keep the rows and use grouping to protect the evaluation split.

In [ ]:
data_checks = pd.Series({
    'Rows in the UCI file': len(email_records),
    'Input features': X.shape[1],
    'Missing fields': int(email_records.isna().sum().sum()),
    'Repeat occurrences of a feature vector': int(X.duplicated().sum()),
    'IDs shared by train and test': len(X_train.index.intersection(X_test.index)),
    'Feature groups shared by train and test': len(set(train_groups) & set(test_groups)),
}, name='count')
display(data_checks.to_frame())
assert email_records.shape == (4601, 58)
assert X.index.equals(y.index) and email_records.index.is_unique
assert set(y) == {0, 1} and not email_records.isna().any().any()
assert not set(train_groups) & set(test_groups)

## Fit one tree, bagged trees, and a baseline forest

Each model sees the same training partition. A single tree fits one set of questions. Bagging fits 100 trees on bootstrap samples and combines their predictions. The baseline forest also fits 100 trees, and draws a random list of candidate features at each split. Using the same number of trees makes that change easier to compare.

In [ ]:
tree_baseline = DecisionTreeClassifier(random_state=seed).fit(X_train, y_train)
bagged_baseline = BaggingClassifier(
    estimator=DecisionTreeClassifier(), n_estimators=100, random_state=seed, n_jobs=1,
).fit(X_train, y_train)
forest_baseline = RandomForestClassifier(
    n_estimators=100, bootstrap=True, max_features='sqrt', random_state=seed, n_jobs=1,
).fit(X_train, y_train)
print(f'Baseline tree depth: {tree_baseline.get_depth()}')
print(f'Trees in the bagged baseline: {len(bagged_baseline.estimators_)}')
print(f'Trees in the forest baseline: {len(forest_baseline.estimators_)}')

## Change one setting at a time

**Joining midway?** The forest is learning to identify spam. We keep one baseline fixed so we can isolate what a single setting changes. Each point fits on three training folds and scores both those fit rows and the remaining validation fold. Rotate the validation fold and average all four scores. The held-out emails stay reserved for the end.

- **Tree depth** limits how many questions a message can follow. Compare the two scores at depth 2, then look for the gap as the tree becomes more flexible. A high training score with much weaker validation is a warning sign for overfitting.
- **Number of trees** controls how many predictions we average. Look for where the validation score starts to level off.
- **Candidate features** limits the measurements a node can start considering. Here `sqrt` gives seven candidates for 57 features, and `None` makes all 57 available. The split search can inspect additional features if needed for a valid split.
- **Parent size** (`min_samples_split`) asks whether the current node has enough samples to attempt splitting.
- **Leaf size** (`min_samples_leaf`) asks whether both resulting children would have enough samples. For example, with 10 rows, parent minimum 8 and leaf minimum 3, a 9/1 proposal fails even though the parent can try to split. A 6/4 proposal passes both minimum-size checks.

Other controls answer different questions. `criterion` chooses how to measure split quality, `max_leaf_nodes` caps the number of terminal nodes, `min_weight_fraction_leaf` sets a minimum share of the total sample weight in a leaf, and `min_impurity_decrease` requires a minimum weighted gain to split.

In [ ]:
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
folds = list(cv.split(X_train, y_train, groups=train_groups))
assert all(not set(train_groups.iloc[fit]) & set(train_groups.iloc[validate])
           for fit, validate in folds)
tuning_metric = 'balanced_accuracy'
tuning_base = RandomForestClassifier(
    n_estimators=40, max_depth=8, max_features='sqrt', random_state=seed, n_jobs=1,
)
sweep_specs = [
    ('max_depth', 'Tree depth', [2, 4, 8, None],
     'Depth changes how many questions a message can follow. Compare the two scores as the limit increases.'),
    ('n_estimators', 'Number of trees', [10, 25, 40, 70],
     'More trees add more predictions to the average. Look for the point where extra trees buy little validation improvement.'),
    ('max_features', 'Candidate features', [1, 'sqrt', None],
     'A smaller candidate list can diversify the trees. It can also hide a useful feature at a split.'),
    ('min_samples_split', 'Parent size', [2, 10, 25, 60],
     'The parent needs this many training samples to attempt a split. Candidate splits still have to meet the child-size limit.'),
    ('min_samples_leaf', 'Leaf size', [1, 4, 12, 24],
     'A split can proceed only if both proposed children meet this minimum. Tiny groups get fewer separate predictions.'),
]
tuning_curves = []
for key, label, values, explanation in sweep_specs:
    train, validation = validation_curve(
        tuning_base, X_train, y_train, param_name=key, param_range=values,
        scoring=tuning_metric, cv=folds, n_jobs=1, error_score='raise',
    )
    tuning_curves.append({
        'key': key,
        'label': label,
        'explanation': explanation,
        'values': [str(value) for value in values],
        'train': train.mean(axis=1).tolist(),
        'validation': validation.mean(axis=1).tolist(),
    })

In [ ]:
# @title Explore the training and validation curves { display-mode: "form" }
from html import escape
from IPython.display import HTML, display


def curve_card(curve, opened=False):
    train, validation = curve['train'], curve['validation']
    labels = curve['values']
    x = [67 + i * 530 / (len(labels) - 1) for i in range(len(labels))]

    def y(score):
        return 244 - (score - 0.5) * 420

    grid = ''.join(
        f'<line x1="57" x2="612" y1="{y(level)}" y2="{y(level)}" stroke="#dce8e3"/>'
        f'<text x="49" y="{y(level) + 5}" text-anchor="end">{level:.0%}</text>'
        for level in (0.5, 0.6, 0.7, 0.8, 0.9, 1.0)
    )
    ticks = ''.join(f'<text x="{position}" y="270" text-anchor="middle">{escape(label)}</text>'
                    for position, label in zip(x, labels))

    def plot(values, color, dash=''):
        points = ' '.join(f'{px:.1f},{y(score):.1f}' for px, score in zip(x, values))
        dots = ''.join(
            f'<circle cx="{px}" cy="{y(score)}" r="5" fill="{color}"><title>{score:.1%}</title></circle>'
            for px, score in zip(x, values)
        )
        return f'<polyline points="{points}" fill="none" stroke="{color}" stroke-width="3" {dash}/>{dots}'

    table = ''.join(f'<tr><td>{escape(label)}</td><td>{fit:.2%}</td><td>{valid:.2%}</td></tr>'
                    for label, fit, valid in zip(labels, train, validation))
    return f"""
      <details name="tuning-curve" {'open' if opened else ''} style="margin:8px 0;background:white;border-radius:8px">
        <summary style="cursor:pointer;padding:8px;font-weight:700">{escape(curve['label'])}</summary>
        <div style="padding:0 10px 10px">
          <p>{escape(curve['explanation'])}</p>
          <svg width="100%" viewBox="0 0 640 285" role="img"
               aria-label="{escape(curve['label'])} versus training and validation balanced accuracy"
               style="max-width:700px;display:block;font:15px system-ui;fill:#273a42">
            {grid}{ticks}{plot(train, '#754723', 'stroke-dasharray="7 5"')}
            {plot(validation, '#14675d')}
          </svg>
          <table style="width:100%;text-align:left"><thead><tr><th>Setting</th><th>Train</th>
            <th>Validation</th></tr></thead><tbody>{table}</tbody></table>
        </div>
      </details>"""


display(HTML('<section data-visual="tuning-curves" style="padding:14px;background:#f2f8f5;border:1px solid #cde1da;'
             'border-radius:12px;color:#253747;font:16px/1.45 system-ui">'
             '<h3 style="margin:.2em 0">Which setting helps on unseen folds?</h3>'
             '<p>Open a setting. Each point is mean balanced accuracy over four folds. '
             '<b style="color:#754723">Train is dashed</b> and <b style="color:#14675d">validation is solid</b>. '
             'The vertical scale is 50% to 100% for every setting. These are cross-validation checks.</p>'
             + ''.join(curve_card(curve, opened=i == 0) for i, curve in enumerate(tuning_curves))
             + '</section>'))

## Compare grid and randomized search

Pair two depth limits with two candidate-feature settings to get four configurations. **Grid search** checks all four. **Randomized search** checks two selected from exactly the same menu. Both use the same estimator settings, four folds and balanced accuracy. Because the inputs are lists and all scores use the same folds, the best grid score is at least as high as the best score from its sampled subset. A larger search space makes the budget choice more consequential.

Ask which cells the randomized search tried, then compare the best scores and number of fits. The broader search after this demonstration is the one that refits and evaluates a final model.

In [ ]:
small_space = {'max_depth': [2, 4], 'max_features': [1, 'sqrt']}
grid_demo = GridSearchCV(
    tuning_base, small_space, cv=folds, scoring=tuning_metric, refit=False, error_score='raise',
).fit(X_train, y_train)
random_demo = RandomizedSearchCV(
    tuning_base, small_space, n_iter=2, cv=folds, scoring=tuning_metric, refit=False,
    random_state=seed, error_score='raise',
).fit(X_train, y_train)

random_choices = random_demo.cv_results_['params']
search_comparison = pd.DataFrame([
    {'Depth': params['max_depth'], 'Candidate features': str(params['max_features']),
     'Mean CV balanced accuracy': score, 'Sampled by random search': params in random_choices}
    for params, score in zip(grid_demo.cv_results_['params'], grid_demo.cv_results_['mean_test_score'])
])
display(search_comparison.round(4))
for name, demo in [('Grid', grid_demo), ('Randomized', random_demo)]:
    candidates = len(demo.cv_results_['params'])
    print(f'{name}: {candidates} configurations × {len(folds)} folds = '
          f'{candidates * len(folds)} fits. Best CV score: {demo.best_score_:.2%}')

In [ ]:
# @title Reveal which configurations each search tried { display-mode: "form" }
def search_card(name, demo, opened=False):
    depth_values = small_space['max_depth']
    feature_values = [str(value) for value in small_space['max_features']]
    tested = {(params['max_depth'], str(params['max_features'])): score
              for params, score in zip(demo.cv_results_['params'], demo.cv_results_['mean_test_score'])}
    best = demo.best_score_
    header = ''.join(f'<th scope="col">Features {escape(value)}</th>' for value in feature_values)
    rows = []
    for depth in depth_values:
        entries = []
        for feature in feature_values:
            score = tested.get((depth, feature))
            if score is None:
                entries.append('<td style="color:#65757a">Skipped</td>')
            elif score == best:
                entries.append(f'<td style="background:#e1f1eb"><strong>{score:.2%} · Best</strong></td>')
            else:
                entries.append(f'<td>{score:.2%}</td>')
        rows.append(f'<tr><th scope="row">Depth {depth}</th>{"".join(entries)}</tr>')
    return f"""
      <details name="tuning-search" {'open' if opened else ''}
               style="background:white;border-radius:8px;margin:8px 0;padding:8px">
        <summary style="cursor:pointer;font-weight:700">{name}</summary>
        <p><strong>{len(tested)} tried × {len(folds)} folds = {len(tested) * len(folds)} fits.</strong>
        Best mean validation balanced accuracy {best:.2%}.</p>
        <table style="width:100%;text-align:left"><thead><tr><th>Tree depth</th>{header}</tr></thead>
          <tbody>{''.join(rows)}</tbody></table>
      </details>"""


display(HTML('<section data-visual="tuning-search" style="padding:14px;background:#f2f8f5;'
             'border:1px solid #cde1da;border-radius:12px;color:#253747;font:16px/1.45 system-ui">'
             '<h3 style="margin:.2em 0">Same search space, different budgets</h3>'
             '<p>Switch between searches to reveal which settings were evaluated.</p>'
             + search_card('Grid search', grid_demo, opened=True)
             + search_card('Randomized search', random_demo) + '</section>'))

## Tune using randomized search on the training partition

Now move from the four-choice comparison to the broader 15-candidate workflow. A hyperparameter is a setting chosen before fitting. The parameter space below provides plausible settings. Randomized search draws a fixed budget of combinations, tests each by stratified grouped four-fold cross-validation, and refits the best-scoring candidate on all training rows. With lists for all parameters, the search samples combinations without replacement. The hold-out set remains separate through this step.

**The five knobs in this lesson**

- `n_estimators` controls the number of trees. More trees stabilize the average of the forest's randomized predictions and cost extra fit time, prediction time, and memory. Improvements commonly level off.
- `max_samples` controls bootstrap draws per tree when bootstrapping is enabled. Fewer draws generally make bags less alike and give each tree less data. Unique row counts depend on repeat draws.
- `max_features` controls the candidate feature budget at each split. A smaller budget can reduce similarity between trees and can make individual questions weaker. The splitter can inspect further features when it needs a valid split.
- `max_depth` caps the depth during growth. A shallower tree often gives simpler, less variable predictions, with a greater risk of missing structure.
- `ccp_alpha` weights the number of leaves in cost-complexity pruning. Larger values favor smaller trees. Choose a value by validation so the forest can still capture useful structure.

These knobs interact. Training and validation scores help reveal whether a setting changes fit, and repeated training runs let you inspect the spread of predictions directly.

In [ ]:
parameter_space = {
    'n_estimators': [64, 100, 140],
    'max_samples': [0.6, 0.8, None],
    'max_features': [1, 'sqrt', None],
    'max_depth': [4, 7, 11, None],
    'ccp_alpha': [0.0, 0.0005, 0.002],
}
search = RandomizedSearchCV(
    RandomForestClassifier(bootstrap=True, oob_score=True, random_state=seed, n_jobs=1),
    parameter_space, n_iter=15, scoring='balanced_accuracy', cv=folds,
    refit=True, return_train_score=True, random_state=seed, n_jobs=1, error_score='raise',
).fit(X_train, y_train)

forest = search.best_estimator_
results = pd.DataFrame(search.cv_results_).sort_values('rank_test_score')
columns = ['rank_test_score', 'mean_train_score', 'mean_test_score', 'std_test_score']
display(results[columns + [f'param_{key}' for key in parameter_space]].head().round(4))
print('Selected settings:', search.best_params_)
print(f'Selected mean CV balanced accuracy: {search.best_score_:.4f}')

## Evaluate the selected forest on the hold-out set

**Accuracy** is the share of all messages classified correctly. **Balanced accuracy** averages recall for the Legitimate and Spam classes. **Spam recall** is the share of recorded spam the model catches. In the confusion matrix, an actual spam message sent to the inbox is a missed spam message. A legitimate message sent to the spam folder is a false alarm.

We used the training folds to select settings. Now all four fitted models get the same untouched emails, and we compare the predictions with their labels. Tuning aims for a better validation score and can still score lower than a baseline on the final test.

In [ ]:
fitted = {'Single tree': tree_baseline, 'Bagged trees': bagged_baseline,
          'Baseline forest': forest_baseline, 'Tuned forest': forest}
test_predictions = {name: model.predict(X_test) for name, model in fitted.items()}
scores = pd.DataFrame.from_dict({name: {
    'accuracy': accuracy_score(y_test, pred),
    'balanced accuracy': balanced_accuracy_score(y_test, pred),
    'Spam recall': recall_score(y_test, pred, pos_label=1),
} for name, pred in test_predictions.items()}, orient='index')
display(scores.round(4))

matrix = confusion_matrix(y_test, test_predictions['Tuned forest'], labels=[0, 1])
display(pd.DataFrame(matrix, index=['Actual Legitimate', 'Actual Spam'],
                     columns=['Predicted Legitimate', 'Predicted Spam']))

In [ ]:
# @title Show the held-out scores side by side
from html import escape
from IPython.display import HTML, display

SPAM_COLOR = '#a74622'
LEGITIMATE_COLOR = '#176959'

def render_panel(content):
    display(HTML('''
    <style>
      .rf-panel { max-width: 1080px; margin: 12px 0; padding: 16px 18px; box-sizing: border-box;
        color: #172c26; background: #f7faf8; border: 1px solid #cfdfd7;
        border-radius: 12px; font: 15px/1.45 system-ui, sans-serif; }
      .rf-panel h3, .rf-panel h4 { color: #18332a; margin: 0 0 8px; }
      .rf-panel p { margin: 6px 0 12px; }
      .rf-panel summary { cursor: pointer; font-weight: 650; padding: 8px 10px; }
      .rf-panel details { background: #fff; border: 1px solid #dae5de;
        margin: 7px 0; border-radius: 7px; }
      .rf-panel details > .rf-detail { padding: 4px 10px 10px; }
      .rf-panel [role="list"] { display: flex; flex-wrap: wrap; gap: 6px 12px; margin: 8px 0; }
      .rf-panel .rf-bar { display: flex; overflow: hidden; border-radius: 6px;
        height: 16px; background: #dce9e2; }
      .rf-panel .rf-score { display: grid; grid-template-columns: minmax(105px, 150px) 1fr 56px;
        gap: 8px; align-items: center; margin: 6px 0; }
      .rf-panel .rf-glyph { display: inline-block; vertical-align: text-bottom; }
      .rf-panel .rf-tiles { display: grid; grid-template-columns: repeat(auto-fit, minmax(20px, 1fr));
        gap: 3px; margin: 8px 0; }
      .rf-panel .rf-tile { display: grid; place-items: center; min-height: 23px;
        border: 1px solid transparent; border-radius: 4px; }
      .rf-panel .rf-tree { width: 100%; height: auto; background: #fff; border-radius: 7px;
        border: 1px solid #dae5de; }
      .rf-panel .rf-scroll { overflow-x: auto; }
      .rf-panel table { width: 100%; border-collapse: collapse; color: #172c26;
        background: #fff; margin: 6px 0; }
      .rf-panel th, .rf-panel td { padding: 4px 6px; text-align: left; border-bottom: 1px solid #dae5de; }
    </style>''' + f'<section class="rf-panel">{content}</section>'))

score_bars = []
for name, score in scores['balanced accuracy'].items():
    score_bars.append(
        f'<div class="rf-score"><span>{escape(name)}</span>'
        f'<div class="rf-bar" aria-hidden="true"><span style="width:{score:.1%};'
        f'background:{LEGITIMATE_COLOR}"></span></div><strong>{score:.2%}</strong></div>'
    )
render_panel(f'''
  <h3>Did the forest improve on the single tree?</h3>
  <p>Every bar uses the same {len(X_test)} held-out emails and a scale from 0% to 100%.
  Balanced accuracy gives Legitimate and Spam equal weight.</p>
  {''.join(score_bars)}
''')

## Read the per-class report

**Spam recall** asks how many recorded spam emails the filter caught. **Spam precision** asks what share of the emails flagged by the filter were recorded as spam. **F1** is the harmonic mean of precision and recall. **Support** counts the emails with each recorded label.

Read the baseline and tuned reports for the same test emails. Compare the missed spam count with the count of legitimate messages incorrectly flagged.

In [ ]:
class_labels = ['Legitimate', 'Spam']
for name in ('Baseline forest', 'Tuned forest'):
    pred = test_predictions[name]
    report = classification_report(y_test, pred, labels=[0, 1], target_names=class_labels,
                                   output_dict=True, zero_division=0)
    per_class = pd.DataFrame(report).T.loc[class_labels, ['precision', 'recall', 'f1-score', 'support']]
    per_class['support'] = per_class['support'].astype(int)
    print(name)
    display(per_class.round(3))
    display(pd.DataFrame(confusion_matrix(y_test, pred, labels=[0, 1]),
                         index=['Actual Legitimate', 'Actual Spam'],
                         columns=['Predicted Legitimate', 'Predicted Spam']))

## Read the winning class from the mean probabilities

The forest class order is `forest.classes_`. Each tree returns scores in the encoded order used during the forest fit. For this 0/1 target those columns match the forest's Legitimate and Spam columns. Average the probability vectors, then choose the corresponding label from `classes_`. The assertion checks that this manual result matches `predict_proba`.

Start with **E574**, the email from the intuition notebook. E574 is in the held-out partition for the reference run. If a different library version changes that partition, the code uses the first held-out email instead.

In [ ]:
example_position = int(X_test.index.get_loc('E574')) if 'E574' in X_test.index else 0
example = X_test.iloc[[example_position]]
mean_prob = np.mean([est.predict_proba(example.to_numpy())[0] for est in forest.estimators_], axis=0)
np.testing.assert_allclose(mean_prob, forest.predict_proba(example)[0])
np.testing.assert_array_equal(forest.classes_[np.argmax(mean_prob)], forest.predict(example)[0])
display(example[exemplar_columns])
display(pd.DataFrame({'class label': forest.classes_, 'mean probability': mean_prob}).round(4))

## Follow one email through the tuned forest

These are unseen test emails. Pick a message and predict its label before revealing the result. Each icon is a fitted tree, and the first three have expandable paths. Compare their leaf values with the forest's mean over **every tree**.

Read a split as “feature value at most the cutoff?” Yes goes to the upper branch and No to the lower branch. At the leaf, the probability assigned to Spam is the share of spam draws there. The dataset's recorded label is revealed separately.

The cards show a few familiar features. The complete tree paths can use any of the 57 inputs. A name such as `word_freq_free` is a percentage of words. A punctuation feature uses the percentage of all characters.

In [ ]:
# @title Open the fitted forest explorer
LEGITIMATE, SPAM = 'Legitimate', 'Spam'
ROUTE_COLOR = {LEGITIMATE: LEGITIMATE_COLOR, SPAM: SPAM_COLOR}
FEATURES = {
    column: ('“' + column.removeprefix('word_freq_') + '” frequency' if column.startswith('word_freq_')
             else column.removeprefix('char_freq_') + ' frequency' if column.startswith('char_freq_')
             else column.removeprefix('capital_run_length_').title() + ' capital run')
    for column in X_train.columns
}

def class_name(label):
    return SPAM if label == 1 else LEGITIMATE

def feature_name(feature_index):
    return FEATURES[X_train.columns[feature_index]]

def tree_icon(color):
    return (f'<svg class="rf-glyph" viewBox="0 0 24 24" width="18" height="18" aria-hidden="true">'
            f'<path d="M12 2l7 9h-3l5 7h-7v4h-4v-4H3l5-7H5z" fill="{color}"/></svg>')

def fitted_tree_diagram(estimator, row, tree_number):
    model = estimator.tree_
    route_nodes = set(estimator.decision_path(row).indices)
    leaf = int(estimator.apply(row)[0])
    last_depth = 4
    positions = {}
    terminal_rows = []

    def locate(node, depth):
        left, right = model.children_left[node], model.children_right[node]
        if left < 0 or depth == last_depth:
            y = 42 + 53 * len(terminal_rows)
            terminal_rows.append(node)
        else:
            y = (locate(left, depth + 1) + locate(right, depth + 1)) / 2
        positions[node] = (16 + 188 * depth, y)
        return y

    locate(0, 0)
    width = max(x for x, y in positions.values()) + 172
    height = 42 + 53 * len(terminal_rows)
    edges, boxes, path_rows = [], [], []
    for node, (x, y) in positions.items():
        left, right = model.children_left[node], model.children_right[node]
        is_leaf = left < 0
        follows = node in route_nodes
        visible_split = left in positions
        if visible_split:
            for child, answer in ((left, 'Yes'), (right, 'No')):
                tx, ty = positions[child]
                on_path = follows and child in route_nodes
                color = LEGITIMATE_COLOR if on_path else '#9dada6'
                edges.append(f'<path d="M{x+156},{y} C{x+176},{y} {tx-20},{ty} {tx},{ty}"'
                             f' stroke="{color}" stroke-width="{3.5 if on_path else 1.2}" fill="none"/>')
                if on_path or node == 0:
                    edges.append(f'<text x="{x+174}" y="{ty-7}" fill="{color}" text-anchor="middle" '
                                 f'font-size="10" font-weight="650">{answer}</text>')
        if is_leaf:
            values = model.value[node][0]
            spam_share = values[1] / values.sum()
            first = 'Reached leaf' if node == leaf else 'Leaf'
            second = f'{spam_share:.1%} Spam'
        elif visible_split:
            first = feature_name(model.feature[node])
            second = f'≤ {model.threshold[node]:.3g}?'
        else:
            first, second = 'More splits below', 'Read the exact path'
        fill = '#dcefe7' if follows else '#f5f7f6'
        stroke = LEGITIMATE_COLOR if follows else '#bcc9c3'
        boxes.append(f'<rect x="{x}" y="{y-20}" width="156" height="40" rx="7" fill="{fill}"'
                     f' stroke="{stroke}" stroke-width="{2 if follows else 1}"/>'
                     f'<text x="{x+9}" y="{y-4}" fill="#18332a" font-size="13" font-weight="700">'
                     f'{escape(first)}</text><text x="{x+9}" y="{y+11}" fill="#18332a" font-size="12">'
                     f'{escape(second)}</text>')

    node = 0
    while node != leaf:
        feature = model.feature[node]
        left, right = model.children_left[node], model.children_right[node]
        went_left = left in route_nodes
        feature_value = row[0, feature]
        path_rows.append(f'<tr><td>{escape(feature_name(feature))}</td><td>{feature_value:.3g}</td>'
                         f'<td>{model.threshold[node]:.3g}</td><td>{"Yes" if went_left else "No"}</td></tr>')
        node = left if went_left else right
    path_table = (
        '<table><thead><tr><th>Feature</th><th>Email value</th><th>Cutoff</th><th>≤ cutoff?</th>'
        f'</tr></thead><tbody>{"".join(path_rows)}</tbody></table>'
        if path_rows else '<p>This tree predicts at the root leaf without asking a split question.</p>'
    )
    return f'''
      <div class="rf-scroll">
      <svg class="rf-tree" viewBox="0 0 {width} {height}" style="min-width:{max(340, int(width*.73))}px"
        role="img" aria-label="Tree {tree_number}. Green outlines mark the path followed by this email.">
        {''.join(edges)}{''.join(boxes)}
      </svg></div>
      <details><summary>Read the exact path</summary><div class="rf-detail">
      <p>Values are rounded for display. The model uses full precision.</p>
      {path_table}</div></details>
    '''

def email_panel(position, heading, opened=False):
    email = X_test.iloc[[position]]
    row = email.to_numpy()
    probabilities = np.stack([est.predict_proba(row)[0] for est in forest.estimators_])
    average = probabilities.mean(axis=0)
    np.testing.assert_allclose(average, forest.predict_proba(email)[0])
    predicted = class_name(forest.classes_[average.argmax()])
    recorded = class_name(y_test.iloc[position])
    tree_classes = [class_name(forest.classes_[p.argmax()]) for p in probabilities]
    tiles = []
    diagrams = []
    for i, (probability, route) in enumerate(zip(probabilities[:, 1], tree_classes), 1):
        shown = i <= 3
        border = 'border-color:#78938a;' if shown else ''
        title = f'Tree {i} · {probability:.1%} Spam'
        tiles.append(f'<span class="rf-tile" style="{border}" title="{title}">'
                     f'{tree_icon(ROUTE_COLOR[route])}</span>')
        if shown:
            diagrams.append(f'''
              <details {'open' if opened and i == 1 else ''}><summary>Tree {i} · {route} · {probability:.1%} Spam</summary>
                <div class="rf-detail">{fitted_tree_diagram(forest.estimators_[i-1], row, i)}</div>
              </details>
            ''')

    inputs = ''.join(f'<span role="listitem"><strong>{escape(FEATURES[column])}</strong> {value:.3g}</span>'
                     for column, value in email[exemplar_columns].iloc[0].items())
    return f'''
      <details name="email-example" {'open' if opened else ''}>
        <summary>{escape(heading)} · test email {email.index[0]}</summary>
        <div class="rf-detail">
          <div role="list" aria-label="Email measurements">{inputs}</div>
          <p><strong>Forest predicts {predicted}.</strong> </p><details><summary>Reveal the recorded label</summary><p>The dataset labels this email <strong>{recorded}</strong>.</p></details>
          <div class="rf-bar" aria-hidden="true">
            <span style="width:{average[0]:.4%};background:{LEGITIMATE_COLOR}"></span>
            <span style="width:{average[1]:.4%};background:{SPAM_COLOR}"></span>
          </div>
          <p><strong>Mean probabilities</strong> · Legitimate {average[0]:.1%} · Spam {average[1]:.1%}</p>
          <p>{tree_icon(LEGITIMATE_COLOR)} {tree_classes.count(LEGITIMATE)} favor Legitimate ·
             {tree_icon(SPAM_COLOR)} {tree_classes.count(SPAM)} favor Spam. These are individual
             preferences. The forest decision uses the probability average above.</p>
          <div class="rf-tiles" role="img" aria-label="{len(tree_classes)} trees. The three outlined
            icons correspond to the diagrams below.">{''.join(tiles)}</div>
          {''.join(diagrams)}
        </div>
      </details>
    '''

spam_scores = forest.predict_proba(X_test)[:, 1]
all_positions = np.arange(len(X_test))
alternatives = all_positions[all_positions != example_position]
near_cutoff = int(alternatives[np.abs(spam_scores[alternatives] - 0.5).argmin()])
remaining = alternatives[alternatives != near_cutoff]
contrast = int(remaining[spam_scores[remaining].argmax() if spam_scores[example_position] < 0.5
                         else spam_scores[remaining].argmin()])
render_panel(
    '<h3>One email, many fitted trees</h3>'
    '<p>Open a email to switch examples. The outlined tree icons have diagrams you can expand. '
    'Hover over any icon to read that tree’s Spam probability.</p>'
    + email_panel(example_position, 'The email from the code cell', opened=True)
    + email_panel(near_cutoff, 'A close decision')
    + email_panel(contrast, 'A contrasting example')
)

## Out-of-bag scoring from a row's eligible trees

Each tree trains on a bootstrap sample. A row is out of bag for a tree when its position appears zero times in that tree's sample. To predict a training row out of bag, average predictions from only the trees that omitted it. Different rows use different groups of trees. With a small forest, some rows may have no eligible trees, so check coverage before interpreting the result.

On the same covered rows, **OOB classification error = 1 − OOB accuracy**. OOB is a convenient estimate inside a bootstrap workflow. Cross-validation lets us compare settings using a shared set of folds. OOB eligibility works row by row. A tree can miss one row and still train on a different row with the same feature values. The grouped held-out set keeps such rows together and remains our final check.

For uniform sampling, a particular row has probability $(1-1/n)^m$ of being omitted in $m$ draws from $n$ rows. For $m=n$ and large $n$, this is about 36.8%. The selected model may use a smaller sample size. The held-out result above uses a separate partition and every tree can predict each held-out row.

In [ ]:
bags = forest.estimators_samples_
in_bag_counts = np.stack([np.bincount(bag, minlength=len(X_train)) for bag in bags])
eligible_counts = (in_bag_counts == 0).sum(axis=0)
eligible_rows = eligible_counts > 0
position = int(np.flatnonzero(eligible_rows)[0])
eligible_trees = np.flatnonzero(in_bag_counts[:, position] == 0)
row = X_train.iloc[[position]].to_numpy()
manual_oob = np.mean([forest.estimators_[i].predict_proba(row)[0] for i in eligible_trees], axis=0)
np.testing.assert_allclose(manual_oob, forest.oob_decision_function_[position])

oob_probs = forest.oob_decision_function_[eligible_rows]
oob_predictions = forest.classes_[oob_probs.argmax(axis=1)]
covered_accuracy = accuracy_score(y_train.to_numpy()[eligible_rows], oob_predictions)
print(f'Training email {X_train.index[position]} is omitted by {len(eligible_trees)} trees.')
print('First eligible tree numbers (1-based):', (eligible_trees[:6] + 1).tolist())
print('That email’s OOB class probabilities:', np.round(manual_oob, 4))
print(f'OOB coverage: {int(eligible_rows.sum())} of {len(eligible_rows)} training rows')
print(f'OOB accuracy on covered rows: {covered_accuracy:.4f}')
print(f'OOB classification error on covered rows: {1 - covered_accuracy:.4f}')
print(f'Built-in OOB accuracy: {forest.oob_score_:.4f}')
if eligible_rows.all():
    np.testing.assert_allclose(covered_accuracy, forest.oob_score_)

In [ ]:
# @title See which training trees can check a email
chosen = list(dict.fromkeys([position, int(eligible_counts.argmin()), int(eligible_counts.argmax())]))
email_groups = []
for i in chosen:
    memberships = in_bag_counts[:, i]
    omitted = int((memberships == 0).sum())
    tiles = []
    for tree_number, count in enumerate(memberships, 1):
        if count == 0:
            color, fill = LEGITIMATE_COLOR, '#ddf1e8'
        else:
            color, fill = '#52655b', '#e5ece8'
        tiles.append(f'<span class="rf-tile" style="color:{color};background:{fill};font-weight:700"'
                     f' title="Tree {tree_number} · {count} draws">{count}</span>')

    if omitted:
        spam_probability = forest.oob_decision_function_[i, 1]
        route = class_name(forest.classes_[forest.oob_decision_function_[i].argmax()])
        prediction = f'<strong>OOB prediction {route}</strong> · {spam_probability:.1%} Spam.'
    else:
        prediction = 'No tree omitted this email, so it cannot receive an OOB prediction.'
    email_groups.append(f'''
      <details name="oob-email" {'open' if i == position else ''}>
        <summary>Training email {X_train.index[i]} · {omitted} eligible trees</summary>
        <div class="rf-detail">
          <div class="rf-tiles" role="img" aria-label="Bootstrap draw counts for {len(memberships)} trees.
            {omitted} have zero draws.">{''.join(tiles)}</div>
          <p>{prediction} Recorded label <strong>{class_name(y_train.iloc[i])}</strong>.</p>
        </div>
      </details>
    ''')
render_panel(
    '<h3>One training email, different bags</h3>'
    '<p>Each tile is one fitted tree. The number counts how often it drew this email into its training bag. '
    'Only the green zeros contribute to the OOB prediction. Open a different training email to see which '
    'trees now qualify.</p>' + ''.join(email_groups)
)

## Explain forest feature importance

Impurity-based importance averages the normalized credits for training impurity reduction across trees that split, then normalizes the average. A tree with splits gives a feature zero credit if none of its splits uses it. Keep that zero when averaging across the trees that split. A split receives credit based on the impurity reduction and the fraction of training samples that reach it. Features with many available split points can receive inflated impurity-based scores.

Permutation importance measures the drop in a chosen score after shuffling one feature on held-out rows. Here the score is balanced accuracy and we repeat each shuffle four times. Redundant or correlated features can share predictive information and look weaker one at a time. Both measures describe this fitted model and evaluation setup.

In [ ]:
permutation = permutation_importance(
    forest, X_test, y_test, scoring='balanced_accuracy', n_repeats=4, random_state=seed, n_jobs=1,
)
importance = pd.DataFrame({
    'impurity importance': forest.feature_importances_,
    'mean held-out score drop': permutation.importances_mean,
    'score-drop spread': permutation.importances_std,
}, index=X.columns).sort_values('mean held-out score drop', ascending=False)
display(importance.round(4))

In [ ]:
# @title Compare the feature importance bars { display-mode: "form" }
importance_bars = []
for title, series in importance[['impurity importance', 'mean held-out score drop']].items():
    bars = []
    scale = max(abs(series)) or 1
    leaders = series.nlargest(10)
    for feature, value in leaders.items():
        extent = abs(value) / scale * 48
        start = 50 - extent if value < 0 else 50
        shade = '#b0673e' if value < 0 else '#207a6e'
        bars.append(
            f'<div style="display:grid;grid-template-columns:minmax(10em,14em) 1fr 4em;'
            f'gap:.5em;align-items:center;margin:6px 0"><span>{escape(FEATURES[feature])}</span>'
            f'<span aria-hidden="true" style="display:block;position:relative;height:.8em;background:#edf4f0">'
            f'<span style="position:absolute;left:50%;height:100%;border-left:1px solid #64766b"></span>'
            f'<span style="position:absolute;left:{start:.4f}%;width:{extent:.4f}%;height:100%;'
            f'background:{shade};border-radius:3px"></span></span><strong>{value:.3f}</strong></div>'
        )
    explanation = ('Share of training split credit assigned to each feature.'
                   if title == 'impurity importance' else
                   'Change in balanced accuracy after shuffling. Negative means the score went up in this run.')
    importance_bars.append(f'<section><h4 style="margin:.8em 0">{escape(title.title())}</h4>'
                           f'<p>{explanation}</p>{"".join(bars)}</section>')
render_panel('<h3>Which email features mattered?</h3>'
             '<p>The charts show the ten highest values for each measure. The table above includes all 57 features. A bar starts at zero. Each chart uses its own scale.</p>'
             + ''.join(importance_bars))

## Work through a pruning decision

`ccp_alpha` sets the cost per leaf. Compare subtrees using **total sample-weighted leaf impurity + alpha × number of leaves**. For example, an illustrative spam tree with four leaves has total impurity 0.12, and pruning a branch leaves two leaves with impurity 0.18.

| Cost per leaf | Keep four leaves | Keep two leaves | Lower-cost option |
|---|---:|---:|---|
| 0.01 | 0.12 + 0.01 × 4 = 0.16 | 0.18 + 0.01 × 2 = 0.20 | Four leaves |
| 0.04 | 0.12 + 0.04 × 4 = 0.28 | 0.18 + 0.04 × 2 = 0.26 | Two leaves |

Higher alpha can make a simpler tree worth a little more impurity on its training rows. Cross-validation helps choose the penalty that works for emails outside those rows.

## Explain the pipeline back in one minute

Fit a tree to predict the recorded label from observable message features. Bagged trees get separately resampled training rows. A random forest also samples a fresh list of candidate columns at each split. Average probabilities to choose a class, then see whether the combined model improves performance on held-out messages.

Cross-validation chooses settings inside the training partition. The held-out set evaluates the models. OOB shows how to reuse the left-out training records. Feature importance helps investigate which signals the fitted model uses.

The lesson site uses exported results from a reproducible run with scikit-learn 1.6.1. Library versions may change the model's exact questions and scores.

[Random Forest API](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.RandomForestClassifier.html) · [RandomizedSearchCV API](https://scikit-learn.org/1.6/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) · [Permutation importance guide](https://scikit-learn.org/1.6/modules/permutation_importance.html)